# **BCDI Pipeline**  
### A Notebook to Run the `BcdiPipeline` Instance  

This notebook provides a structured workflow for running a **Bragg Coherent Diffraction Imaging (BCDI) pipeline**.  

The `BcdiPipeline` class handles the entire process, including:  
- **Pre-processing** → Data preparation and corrections.  
- **Phase retrieval** → Running PyNX algorithms to reconstruct the phase.  
- **Post-processing** → Refining, analysing (get the strain!), and visualising results.  

You can provide **either**:  
- A **YAML parameter file** for full automation.  
- A **Python dictionary** for interactive control in this notebook.  


In [ ]:
# import required packages
import os
import cdiutils  # core library for BCDI processing


In [ ]:
# Path to your existing PyNX environment
PYNX_ENV = "/projects/htp-saxs/bcdi/davidyang25/conda-envs/pynx-env"
PYNX_BIN = os.path.join(PYNX_ENV, "bin")

# Make PyNX executables visible to CDIutils
os.environ["PATH"] = PYNX_BIN + os.pathsep + os.environ.get("PATH", "")

## **General Parameters**
Here, define the key parameters for **accessing and saving data** before running the pipeline.  
- **These parameters must be set manually by the user** before execution.  
- The output data will be saved in a structured directory format based on `sample_name` and `scan`. However, you can change the directory path if you like.


In [ ]:
# define the key parameters (must be filled in by the user)
beamline_setup: str = "p10"  # example: "ID01" (provide the beamline setup)
experiment_data_dir_path: str = "/projects/htp-saxs/bcdi/davidyang25/beamtime/DESY/P10/I-20240156/data/raw"
sample_name: str = "MC811_insitu_1"  # example: "Sample_Pt" (specify the sample name)
scan: int = 95  # example: 42 (specify the scan number)

# choose where to save the results (default: current working directory)
dump_dir = "/projects/htp-saxs/bcdi/davidyang25/analysis/DESY/P10/I-20260769/cdiutils" + f"/results/{sample_name}/S{scan}/"

# load the parameters and parse them into the BcdiPipeline class instance
params = cdiutils.pipeline.get_params_from_variables(dir(), globals())
bcdi_pipeline = cdiutils.BcdiPipeline(params=params)

## **Pre-Processing**  

If you need to update specific parameters, you can **pass them directly** into the `preprocess` method.  

### **Main Parameters**
- `preprocess_shape` → The shape of the cropped window used throughout the processes.  
  - Can be a **tuple of 2 or 3 values**.  
  - If only **2 values**, the entire rocking curve is used.  

- `voxel_reference_methods` → A `list` (or a single value) defining how to centre the data.  
  - Can include `"com"`, `"max"`, or a `tuple` of `int` (specific voxel position).  
  - Example:
    ```python
    voxel_reference_methods = [(70, 200, 200), "com", "com"]
    ```
    - This centres a box of size `preprocess_shape` around `(70, 200, 200)`, then iteratively refines it using `"com"` (only computed within this box).
    - Useful when `"com"` fails due to artifacts or `"max"` fails due to hot pixels.  
    - Default: `["max", "com", "com"]`.  

- `rocking_angle_binning` → If you want to bin in the **rocking curve direction**, provide a binning factor (ex.: `2`).  

- `light_loading` → If `True`, loads only the **ROI of the data** based on `voxel_reference_methods` and `preprocess_output_shape`.  

- `hot_pixel_filter` → Removes isolated hot pixels. 
  - Default: `False`.  

- `background_level` → Sets the background intensity to be removed.  
  - Example: `3`.  
  - Default: `None`.  

- `hkl` → Defines the **Bragg reflection** measured to extend *d*-spacing values to the lattice parameter.  
  - Default: `[1, 1, 1]`.  


In [ ]:
bcdi_pipeline.preprocess(
    preprocess_shape=(100, 312, 312),  # define cropped window size
    voxel_reference_methods=[(50, 900, 850),"com"],  # centring method sequence
    # voxel_reference_methods=[(50, 900, 850)],  # centring method sequence for light loading
    hot_pixel_filter=True,  # remove isolated hot pixels
    background_level=6,  # background intensity level to remove
    light_loading=False,  # use light loading to save memory upon loading
    detector_name = "e4m",
    rocking_angle_binning = 1,
    detector_binning = (2, 2), # use 2 by 2 binning post crop
    hkl = [0, 0, 3],
    det_calib_params={
        "distance": 1.839,  # direct beam position vertical,
        "cch1": 1031,  # horizontal
        "cch2": 1339,  # detector pixel size in m
        "pwidth1": 7.5e-05,  # detector pixel size in m
        "pwidth2": 7.5e-05,  # sample to detector distance in m
        "detrot": 0,
        "tilt": 0,
        "tiltazimuth": 0,
    },
)

## **[Machine learning assisted masking of parasitic signals in Bragg coherent diffraction imaging](https://doi.org/10.1107/S160057752501152X)**

The `remove_aliens` method identifies parasitic diffraction spots (aliens) in the preprocessed detector data using intensity thresholding and DBSCAN clustering. It allows you to interactively select which clusters should be masked. Based on [alienclustering](https://github.com/ewbellec/alienclustering) by Ewen Bellec.

Run this method **after `preprocess()` and before phase retrieval**. Do not rerun `preprocess()` after applying the alien mask, as preprocessing will regenerate the detector mask and overwrite the changes.

### **Main Parameters**

- `auto_mask_aliens` → Enables or disables automatic alien detection and masking.
  - Default: `False`.

- `threshold` → Defines the intensity threshold used to identify potential alien diffraction spots.
  - Higher values select only brighter features.
  - Lower values include weaker intensity features.
  - Default: `0.5`.

- `nb_clusters_kept` → Maximum number of candidate clusters displayed for selection.
  - Clusters are ranked according to the `sorting` parameter.
  - Default: `10`.

- `sorting` → Defines how candidate clusters are ranked.
  - `"size"`: Ranks clusters by the number of voxels.
  - `"asym"`: Ranks clusters by their average asymmetry.
  - `"max"`: Ranks clusters by their maximum intensity.
  - `"None"`: Disables sorting.
  - Default: `"size"`.

- `remove_hot_pixels` → If `True`, applies hot-pixel filtering to the detector intensity data before intensity thresholding and DBSCAN clustering.
  - This modifies the intensity data used for alien detection.
  - **Important:** Hot pixels removed during this step are NOT automatically added to the final alien detector mask. The final alien mask contains only the clusters manually selected in the interactive selector.
  - If you want to mask hot pixels in the original detector data, they must be identified and added to the detector mask separately in `preprocess()`.
  - Default: `False`.

- `filter_small_values` → If `True`, sets intensity values below one to zero during preprocessing to improve cluster detection.
  - Default: `True`.

- `plot_intensity_thr` → Controls whether diagnostic projections of the intensity-threshold mask are displayed.
  - The plot is displayed before the interactive cluster-selection interface.
  - Default: `True`.

- `plot_alien_mask` → Controls whether the newly generated alien mask is plotted after the user confirms the cluster selection.
  - This displays the mask generated from the selected clusters, not the final combined detector mask.
  - Default: `True`.

- `smooth_mask` → If `True`, applies maximum-filter smoothing to the intensity-threshold mask before clustering.
  - Helps connect nearby voxels belonging to the same diffraction feature, but excessive smoothing can merge alien spots with the central Bragg peak.
  - Default: `True`.

- `smoothing_size` → Defines the filter size used for mask smoothing.
  - Larger values produce more aggressive smoothing.
  - Default: `2`.

- `inspect_mask` → If `True`, opens an interactive slice viewer of the final combined detector mask after the selected alien clusters have been added and the updated mask has been saved.
  - Default: `False`.

### **Interactive Cluster Selection**

After detection, the candidate clusters are displayed in the notebook. Select the clusters corresponding to parasitic diffraction spots using the checkboxes, then click **Confirm Selection** to apply the mask.

- The selected alien clusters are added to the existing detector mask and existing masked pixels are preserved.
- The updated mask is saved to the PyNX input mask (`S*_pynx_input_mask.npz`) and the preprocessed CXI file.
- A separate alien mask is also saved as `S*_alien_mask.npz`.

In [ ]:
auto_mask_aliens=True

bcdi_pipeline.remove_aliens(
    auto_mask_aliens=auto_mask_aliens,
    threshold=0.3,
    nb_clusters_kept=3,
    sorting="size",
    remove_hot_pixels=False,
    filter_small_values=True,
    plot_intensity_thr=False,
    smooth_mask=True,
    smoothing_size=3,
    inspect_mask=True,
    plot_alien_mask=True,
)

## **[PyNX](https://pynx.esrf.fr/en/latest/index.html) Phase Retrieval**
See the [pynx.cdi](https://pynx.esrf.fr/en/latest/scripts/pynx-cdi-id01.html) documentation for details on the phasing algorithms used here.  

**Algorithm recipe**

You can either:  
- provide the exact chain of algorithms.  
- or specify the number of iterations for **RAAR**, **HIO**, and **ER**.  

```python
algorithm = None  # ex: "(Sup * (ER**20)) ** 10, (Sup*(HIO**20)) ** 15, (Sup*(RAAR**20)) ** 25"
nb_raar = 500
nb_hio =  300
nb_er =  200
psf = "pseudo-voigt,1,0.05,20"
```
**Support-related parameters**
```python
support = "auto"  # ex: bcdi_pipeline.pynx_phasing_dir + "support.cxi" (path to an existing support)
```
>_Note: If strain seems to large, don't use "auto" (autocorrelation) but use "circle" or "square", in combination with "support_size"_  
```python
support_threshold = "0.15, 0.40"  # must be a string
support_update_period = 20
support_only_shrink = False
support_post_expand = None  # ex: "-1,1" or "-1,2,-1"
support_update_border_n = None
support_smooth_width_begin = 2
support_smooth_width_end = 0.5
```
**Other parameters**
```python
positivity = False
beta = 0.9  # β parameter in HIO and RAAR
detwin = True
rebin = "1, 1, 1"  # must be a string
```
**Number of Runs & Reconstructions to Keep**
```python
nb_run = 20  # total number of runs
nb_run_keep = 10  # number of reconstructions to keep
```

**Override defaults in `phase_retrieval`**

You can override any default parameter directly in the phase_retrieval method:
```python
bcdi_pipeline.phase_retrieval(nb_run=50, nb_run_keep=25)
```
If a parameter is not provided, the default value is used.

### **Phase Retrieval GUI**
You can also launch a **Graphical User Interface (GUI)** to interactively set parameters and run phase retrieval.  
```python
bcdi_pipeline.phase_retrieval_gui()
```
In that case, you can take care of the the result analysis, the selection of the best reconstructions, and the mode decomposition. Then, simply jump to the post-processing step cell.

In [ ]:
bcdi_pipeline.phase_retrieval(
    clear_former_results=True,
    nb_run=50,
    nb_run_keep=25,
    # nb_raar =  0,
    # nb_hio =  600,
    # nb_er =  200,
    # algorithm = " (Sup*(ER**20)) ** 10, (Sup * (HIO**20)) ** 30",
    # support = bcdi_pipeline.pynx_phasing_dir + "support.cxi",
    support_threshold = "0.27, 0.36",
    support_update_period = 20, # set to 0 for fixed support
)

### **Analyse the phasing results**

This step evaluates the quality of the phase retrieval results by sorting reconstructions based on a `sorting_criterion`.  

##### **Available Sorting Criteria**
- `"mean_to_max"` → Difference between the mean of the **Gaussian fit of the amplitude histogram** and its maximum value. A **smaller difference** indicates a more homogeneous reconstruction.  
- `"sharpness"` → Sum of the amplitude within the support raised to the power of 4. **Lower values** indicate greater homogeneity.  
- `"std"` → **Standard deviation** of the amplitude.  
- `"llk"` → **Log-likelihood** of the reconstruction.  
- `"llkf"` → **Free log-likelihood** of the reconstruction.  


In [ ]:
bcdi_pipeline.analyse_phasing_results(
    sorting_criterion="mean_to_max",  # selects the sorting method
    # sorting_criterion="sharpness",  # selects the sorting method
    nb_to_plot = 15,
    # Optional parameters
    # plot_phasing_results=False,  # uncomment to disable plotting
    # plot_phase=True,  # uncomment to enable phase plotting
)

### **Optionally, generate a support for further phasing attempts**  

##### **Parameters**
- `run` → set to either:  
  - `"best"` to use the best reconstruction.  
  - an **integer** corresponding to the specific run you want.  
- `output_path` → the location to save the generated support. By default, it will be saved in the `pynx_phasing` folder.  
- `fill` →  whether to fill the support if it contains holes.  
  - Default: `False`.
- `verbose` → whether to print logs and display a plot of the support.  

<!-- ```python
bcdi_pipeline.generate_support_from(run="best", output_path="path/to/save", verbose=True)
``` -->

In [ ]:
# # bcdi_pipeline.generate_support_from("best", fill=True, verbose=True)  # uncomment to generate a support
# bcdi_pipeline.generate_support_from(31, fill=True, verbose=True)  # uncomment to generate a support

### **Selection of the best reconstructions & mode decomposition**

You can select the best reconstructions based on a **sorting criterion** and keep a specified number of top candidates.  

##### **Parameters**
- `nb_of_best_sorted_runs` → the number of best reconstructions to keep, selected based on the `sorting_criterion` used in the `analyse_phasing_results` method above.  
- `best_runs` → instead of selecting based on sorting, you can manually specify a list of reconstruction numbers.

By default, the **best reconstructions** are automatically selected.  

Once the best candidates are chosen, `mode_decomposition` analyses them to extract dominant features.  



In [ ]:
# define how many of the best candidates to keep
number_of_best_candidates: int = 5

# select the best reconstructions based on the sorting criterion
bcdi_pipeline.select_best_candidates(
    nb_of_best_sorted_runs=number_of_best_candidates
    # best_runs=[10]  # uncomment to manually select a specific run
)

# perform mode decomposition on the selected reconstructions
bcdi_pipeline.mode_decomposition()

## **Post-processing**

This stage includes several key operations:  
- **orthogonalisation** of the reconstructed data.  
- **phase manipulation**:  
  - phase unwrapping  
  - phase ramp removal  
- **computation of physical properties**:  
  - displacement field  
  - strain  
  - d-spacing  
- **visualisation**: Generate multiple plots for analysis.  


In [ ]:
bcdi_pipeline.postprocess(
    isosurface=0.05,  # threshold for isosurface
    # voxel_size=(10,10,10),  # use default voxel size if not provided. otherwise set it in nm, e.g., (10,10,10)
    flip=False,  # whether to flip the reconstruction if you got the twin image (enantiomorph)
    handle_defects=True,
    amplitude_morph_close=False, # set to True to close holes in the amplitude due to defects
    amplitude_morph_close_size=10, # the size to fill, as the length of a cube
    output_shape=None, # crops/pads the array size of the output reconstruction, e.g., (128, 128, 128)
)

## **3D interactive plot**

Display an interactive 3D isosurface of the final reconstruction and explore different quantities for colouring.

What you can do
- Visualise an isosurface of the reconstructed object (amplitude / support).
- Colour the surface by different quantities: amplitude, phase, displacement, strain, d-spacing, etc.
- Interactively adjust:
    - isosurface threshold (isosurface level)
    - colormap and value range
- Rotate, zoom and pan the scene with the mouse; use the toolbar to reset view or save a screenshot.


In [ ]:
bcdi_pipeline.show_3d_final_result()

## **Feedback & Issue Reporting**  

If you have **comments, suggestions, or encounter any issues**, please reach out:  

📧 **Email:** [clement.atlan@esrf.fr](mailto:clement.atlan@esrf.fr?subject=cdiutils)  
🐙 **GitHub Issues:** [Report an issue](https://github.com/clatlan/cdiutils/issues)  


## Credits
This notebook was created by Clément Atlan, ESRF, 2025. It is part of the `cdiutils` package, which provides tools for BCDI data analysis and visualisation.
If you have used this notebook or the `cdiutils` package in your research, please consider citing the package https://github.com/clatlan/cdiutils/
You'll find the citation information in the `cdiutils` package documentation.

```bibtex
@software{Atlan_Cdiutils_A_python,
author = {Atlan, Clement},
doi = {10.5281/zenodo.7656853},
license = {MIT},
title = {{Cdiutils: A python package for Bragg Coherent Diffraction Imaging processing, analysis and visualisation workflows}},
url = {https://github.com/clatlan/cdiutils},
version = {0.2.0}
}
```

